In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

from pyspark.sql.functions import*
from pyspark.sql.types import *
from pyspark.sql import Window
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [0]:
# STARTER CODE - DO NOT EDIT THIS CELL

#from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

customSchema = StructType([
    StructField("lpep_pickup_datetime", StringType(), True),
    StructField("lpep_dropoff_datetime", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("payment_type", IntegerType(), True)
])

In [0]:
# STARTER CODE - YOU CAN LOAD ANY FILE WITH A SIMILAR SYNTAX.
# Correct file path for Databricks File System (update if you are using a different volume or file name)
file_path = "/Volumes/workspace/default/q2vol/nyc-tripdata.csv"

# Read the CSV file using Spark DataFrame
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(df.limit(5))


lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,payment_type
12/21/2018 15:17,12/21/2018 15:18,264,264,5,0.0,3.0,2
01/01/2019 0:10,01/01/2019 0:16,97,49,2,0.86,6.0,2
01/01/2019 0:27,01/01/2019 0:31,49,189,2,0.66,4.5,1
01/01/2019 0:46,01/01/2019 1:04,189,17,2,2.68,13.5,1
01/01/2019 0:19,01/01/2019 0:39,82,258,1,4.53,18.0,2


In [0]:
# LOAD THE "taxi_zone_lookup.csv" FILE SIMILARLY AS ABOVE. CAST ANY COLUMN TO APPROPRIATE DATA TYPE IF NECESSARY.

#ENTER THE CODE BELOW

# "LocationID","Borough","Zone","service_zone"
taxi_zone_lookup_path = '/Volumes/workspace/default/q2vol/taxi_zone_lookup.csv'

taxi_zone_lookup_schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])
taxi_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(taxi_zone_lookup_path)

display(taxi_df.limit(5))


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone


In [0]:
#// STARTER CODE 
# // Some commands that you can use to see your dataframes and results of the operations. You can comment the df.show(5) and uncomment display(df) to see the data differently. You will find these two functions useful in reporting your results.
df.show(5)
#display(df.limit(5))

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|    12/21/2018 15:17|     12/21/2018 15:18|         264|         264|              5|          0.0|        3.0|           2|
|     01/01/2019 0:10|      01/01/2019 0:16|          97|          49|              2|         0.86|        6.0|           2|
|     01/01/2019 0:27|      01/01/2019 0:31|          49|         189|              2|         0.66|        4.5|           1|
|     01/01/2019 0:46|      01/01/2019 1:04|         189|          17|              2|         2.68|       13.5|           1|
|     01/01/2019 0:19|      01/01/2019 0:39|          82|         258|              1|         4.53|       18.0|      

In [0]:
# // STARTER CODE - DO NOT EDIT THIS CELL
# Filter the data to only keep the rows where "PULocationID" and the "DOLocationID" are different and the "trip_distance" is strictly greater than 2.0 (>2.0).

# VERY VERY IMPORTANT: ALL THE SUBSEQUENT OPERATIONS MUST BE PERFORMED ON THIS FILTERED DATA

df_filter = df.filter((df.PULocationID != df.DOLocationID) & (df.trip_distance > 2.0))
df_filter.show(5)

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|lpep_pickup_datetime|lpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|payment_type|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+------------+
|     01/01/2019 0:46|      01/01/2019 1:04|         189|          17|              2|         2.68|       13.5|           1|
|     01/01/2019 0:19|      01/01/2019 0:39|          82|         258|              1|         4.53|       18.0|           2|
|     01/01/2019 0:47|      01/01/2019 1:00|         255|          33|              1|         3.77|       13.5|           1|
|     01/01/2019 0:12|      01/01/2019 0:30|          76|         225|              1|          4.1|       16.0|           1|
|     01/01/2019 0:16|      01/01/2019 0:39|          25|          89|              1|         7.75|       25.5|      

In [0]:
# PART 1a: The top-5 most popular drop locations - "DOLocationID", sorted in descending order - if there is a tie, then one with lower "DOLocationID" gets listed first

# Output Schema: DOLocationID int, number_of_dropoffs int 

# Hint: Checkout the groupBy(), orderBy() and count() functions.

# ENTER THE CODE BELOW
DOLocationID_name = 'DOLocationID'
number_of_dropoffs_name = 'number_of_dropoffs'
df_1a = df_filter.groupBy(DOLocationID_name).agg(count(DOLocationID_name).alias(number_of_dropoffs_name)).orderBy(desc(number_of_dropoffs_name), asc(DOLocationID_name))
df_1a.show()
display(df_1a)

+------------+------------------+
|DOLocationID|number_of_dropoffs|
+------------+------------------+
|          61|              5937|
|         138|              5146|
|         239|              4133|
|         244|              4006|
|          42|              3859|
|         181|              3854|
|         129|              3838|
|         142|              3784|
|         236|              3784|
|         238|              3760|
|         225|              3697|
|          17|              3527|
|         140|              3426|
|          76|              3404|
|         226|              3236|
|         237|              3233|
|          89|              3216|
|         188|              3128|
|          75|              3027|
|           7|              2952|
+------------+------------------+
only showing top 20 rows


DOLocationID,number_of_dropoffs
61,5937
138,5146
239,4133
244,4006
42,3859
181,3854
129,3838
142,3784
236,3784
238,3760


In [0]:
# PART 1b: The top-5 most popular pickup locations - "PULocationID", sorted in descending order - if there is a tie, then one with lower "PULocationID" gets listed first 

# Output Schema: PULocationID int, number_of_pickups int

# ENTER THE CODE BELOW



In [0]:
# PART 2: List the top-3 locations with the maximum overall activity, i.e. sum of all pickups and all dropoffs at that LocationID. In case of a tie, the lower LocationID gets listed first.

# Output Schema: LocationID int, number_activities int

# Hint: In order to get the result, you may need to perform a join operation between the two dataframes that you created in earlier parts (to come up with the sum of the number of pickups and dropoffs on each location). 

# ENTER THE CODE BELOW



In [0]:
# PART 3: List all the boroughs (including "Unknown" and "EWR") in the order of having the highest to lowest number of activities (i.e. sum of all pickups and all dropoffs at that LocationID), along with the total number of activity counts for each borough in NYC during that entire period of time.

# Output Schema: Borough string, total_number_activities int

# Hint: You can use the dataframe obtained from the previous part, and will need to do the join with the 'taxi_zone_lookup' dataframe. Also, checkout the "agg" function applied to a grouped dataframe.

# ENTER THE CODE BELOW



In [0]:
# PART 4: List the top 2 days of week with the largest number of daily average pickups, along with the average number of pickups on each of the 2 days in descending order (no rounding off required). Here, the average pickup is calculated by taking an average of the number of pick-ups on different dates falling on the same day of the week. For example, 02/01/2021, 02/08/2021 and 02/15/2021 are all Mondays, so the average pick-ups for these is the sum of the pickups on each date divided by 3.

# Note: The day of week is a string of the day’s full spelling, e.g., "Monday" instead of the number 1 or "Mon". Also, the pickup_datetime is in the format: MM/dd/yyyy H:mm.

# Output Schema: day_of_week string, avg_count float

# Hint: You may need to group by the "date" (without time stamp - time in the day) first. Checkout "date_format" and "to_date" functions.

# ENTER THE CODE BELOW



In [0]:
# PART 5: For each particular hour of a day (0 to 23, 0 being midnight) - in their order from 0 to 23, find the zone in Brooklyn borough with the LARGEST number of pickups. 

# Note: All dates for each hour should be included.

# Output Schema: hour_of_day int, zone string, max_count int

# Hint: You may need to use "Window" over hour of day, along with "group by" to find the MAXIMUM count of pickups

# ENTER THE CODE BELOW


In [0]:
# PART 6 - Find which 3 different days in the month of January, in Manhattan, saw the largest positive percentage increase in pick-ups compared to the previous day, in the order from largest percentage increase to smallest percentage increase 

# Note: All years need to be aggregated to calculate the pickups for a specific day of January. The change from Dec 31 to Jan 1 can be excluded.

# Output Schema: day int, percent_change float

# Hint: You might need to use lag function, over a window ordered by day of month.

# ENTER THE CODE BELOW

